# MRI data summary

Summarize subjects and functional MRI run quality from the MRIQC CSV.

- **Patient:** subject number starts with `1` (after removing `sub-`).
- **HC:** subject number starts with `3`.
- Each subject is counted once across all sessions and tasks.
- Poor-quality counts include all observed subject × session × task combinations, including those with zero flagged runs. Absent combinations are not interpreted as zero runs.

Run all cells to refresh the summaries. Only pandas is required in addition to Jupyter.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

CSV_PATH = Path('/Users/xiaoqianxiao/projects/IFOCUS/derivatives/qc/mriqc/QC_func.csv')
qc = pd.read_csv(CSV_PATH, dtype='string')
required = ['subID', 'sesID', 'taskID', 'runID', 'Poor_Quality']
missing = sorted(set(required) - set(qc.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
for column in required:
    qc[column] = qc[column].str.strip()
if qc[required].isna().any().any() or qc[required].eq('').any().any():
    raise ValueError('Required identifiers and Poor_Quality must not be missing or empty.')

subject_number = qc['subID'].str.replace(r'^sub-', '', regex=True)
qc['group'] = subject_number.str[0].map({'1': 'Patient', '3': 'HC'}).fillna('Unclassified')
flag_map = {'true': True, 'false': False, '1': True, '0': False}
flags = qc['Poor_Quality'].str.lower()
if not flags.isin(flag_map).all():
    invalid = qc.loc[~flags.isin(flag_map), 'Poor_Quality'].unique().tolist()
    raise ValueError(f'Unrecognized Poor_Quality values: {invalid}')
qc['poor_quality'] = flags.map(flag_map).astype(bool)
run_key = ['subID', 'sesID', 'taskID', 'runID']
if qc.duplicated(run_key).any():
    display(qc.loc[qc.duplicated(run_key, keep=False), required])
    raise ValueError('Duplicate run identifiers found; resolve duplicates before counting runs.')
print(f'Loaded {len(qc):,} runs from {CSV_PATH.name}.')


Loaded 1,047 runs from QC_func.csv.


## 1. Unique subjects in each group

Counts describe subjects represented in this CSV, before quality exclusions. Unexpected subject prefixes are listed as Unclassified.

In [2]:
group_order = ['Patient', 'HC']
if qc['group'].eq('Unclassified').any():
    group_order.append('Unclassified')
subject_counts = (
    qc.groupby('group')['subID'].nunique()
    .reindex(group_order, fill_value=0)
    .rename('n_subjects').rename_axis('group').reset_index()
)
display(subject_counts)
print(f"Total unique subjects: {qc['subID'].nunique()}")


Total unique subjects: 85


,group,n_subjects
0,Patient,42
1,HC,43


## 2. Poor-quality runs per subject × session × task

`n_poor_quality_runs` counts runs where `Poor_Quality` is true. The table also provides total runs, the percentage flagged, and flagged run IDs. All rows are displayed.

In [3]:
summary_keys = ['group', 'subID', 'sesID', 'taskID']
run_summary = (
    qc.groupby(summary_keys, sort=True)
    .agg(n_runs=('runID', 'size'), n_poor_quality_runs=('poor_quality', 'sum'))
    .reset_index()
)
poor_run_ids = (
    qc.loc[qc['poor_quality']].groupby(summary_keys)['runID']
    .agg(lambda values: ', '.join(sorted(values)))
    .rename('poor_quality_run_ids').reset_index()
)
run_summary = run_summary.merge(poor_run_ids, on=summary_keys, how='left', validate='one_to_one')
run_summary['poor_quality_run_ids'] = run_summary['poor_quality_run_ids'].fillna('')
run_summary['poor_quality_percent'] = (100 * run_summary['n_poor_quality_runs'] / run_summary['n_runs']).round(2)
run_summary = run_summary.sort_values(['subID', 'sesID', 'taskID']).reset_index(drop=True)
assert run_summary['n_runs'].sum() == len(qc)
assert run_summary['n_poor_quality_runs'].sum() == qc['poor_quality'].sum()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(run_summary)
print(f"Total poor-quality runs: {qc['poor_quality'].sum()} / {len(qc)}")


Total poor-quality runs: 43 / 1047


,group,subID,sesID,taskID,n_runs,n_poor_quality_runs,poor_quality_run_ids,poor_quality_percent
0,Patient,sub-102,ses-T12,rest,2,0,,0.0
1,Patient,sub-102,ses-T12,selfother,2,0,,0.0
2,Patient,sub-102,ses-T6,rest,2,0,,0.0
3,Patient,sub-102,ses-T6,selfother,2,0,,0.0
4,Patient,sub-102,ses-baseline,rest,2,0,,0.0
5,Patient,sub-102,ses-baseline,selfother,2,0,,0.0
6,Patient,sub-102,ses-repeatbaseline,rest,2,0,,0.0
7,Patient,sub-102,ses-repeatbaseline,selfother,2,0,,0.0
8,Patient,sub-104,ses-T12,rest,2,0,,0.0
9,Patient,sub-104,ses-T12,selfother,2,0,,0.0


### Combinations with at least one poor-quality run

A filtered view of the same summary for easier review.

In [4]:
poor_quality_summary = run_summary.loc[run_summary['n_poor_quality_runs'] > 0].copy()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(poor_quality_summary)


,group,subID,sesID,taskID,n_runs,n_poor_quality_runs,poor_quality_run_ids,poor_quality_percent
10,Patient,sub-104,ses-T6,rest,2,2,"01, 02",100.0
11,Patient,sub-104,ses-T6,selfother,2,2,"01, 02",100.0
20,Patient,sub-105,ses-baseline,rest,2,1,01,50.0
21,Patient,sub-105,ses-baseline,selfother,2,1,02,50.0
44,Patient,sub-110,ses-repeatbaseline,rest,2,1,01,50.0
45,Patient,sub-110,ses-repeatbaseline,selfother,2,2,"01, 02",100.0
77,Patient,sub-121,ses-T6,selfother,2,1,02,50.0
78,Patient,sub-121,ses-baseline,rest,2,2,"01, 02",100.0
81,Patient,sub-121,ses-repeatbaseline,selfother,2,1,01,50.0
122,Patient,sub-132,ses-T12,rest,2,1,01,50.0


## 3. Subject session status

Patients are expected to have **baseline, repeatbaseline, t6, t12**; HCs are expected to have **baseline, repeatbaseline, t12**. Session labels are matched without regard to case and with the `ses-` prefix removed.

A session is present if at least one run appears in this CSV, regardless of quality or task completeness. Only subjects represented in the CSV can be assessed. The tables list complete subjects and every possible missing-session combination (including combinations with zero subjects), followed by individual subject status. Unexpected sessions are reported separately and do not replace required sessions.

In [5]:
from itertools import combinations

expected_sessions = {
    'Patient': ('baseline', 'repeatbaseline', 't6', 't12'),
    'HC': ('baseline', 'repeatbaseline', 't12'),
}
session_data = qc.assign(session=qc['sesID'].str.lower().str.replace(r'^ses-', '', regex=True))
subject_rows = []
for (group, subject), records in session_data.groupby(['group', 'subID'], sort=True):
    observed = set(records['session'])
    expected = expected_sessions.get(group)
    missing_sessions = tuple(session for session in expected if session not in observed) if expected else ()
    subject_rows.append({
        'group': group,
        'subID': subject,
        'status': ('Missing sessions' if missing_sessions else 'All sessions') if expected else 'Unknown group',
        'n_expected_sessions': len(expected) if expected else pd.NA,
        'n_present_expected_sessions': len(set(expected) & observed) if expected else pd.NA,
        'present_sessions': ', '.join(sorted(observed)),
        'missing_sessions': ', '.join(missing_sessions) if expected else 'Not assessed',
        'unexpected_sessions': ', '.join(sorted(observed - set(expected))) if expected else 'Not assessed',
    })
subject_session_status = pd.DataFrame(subject_rows, columns=[
    'group', 'subID', 'status', 'n_expected_sessions', 'n_present_expected_sessions',
    'present_sessions', 'missing_sessions', 'unexpected_sessions',
])

combination_rows = []
for group, expected in expected_sessions.items():
    for n_missing in range(len(expected) + 1):
        for missing_combination in combinations(expected, n_missing):
            missing_label = ', '.join(missing_combination)
            subjects = sorted(subject_session_status.loc[
                subject_session_status['group'].eq(group)
                & subject_session_status['missing_sessions'].eq(missing_label), 'subID'
            ].tolist())
            combination_rows.append({
                'group': group,
                'status': 'All sessions' if n_missing == 0 else 'Missing sessions',
                'n_missing_sessions': n_missing,
                'missing_sessions': missing_label or 'None',
                'n_subjects': len(subjects),
                'subjects': ', '.join(subjects) or '—',
            })
session_combination_summary = pd.DataFrame(combination_rows)
session_status_counts = (
    session_combination_summary.groupby(['group', 'status'])['n_subjects'].sum()
    .unstack('status', fill_value=0).reindex(['Patient', 'HC'])
    .reindex(columns=['All sessions', 'Missing sessions'], fill_value=0)
)
session_status_counts['Total subjects'] = session_status_counts.sum(axis=1)
for group in expected_sessions:
    assert session_status_counts.loc[group, 'Total subjects'] == qc.loc[qc['group'].eq(group), 'subID'].nunique()

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    print('Subject counts by session status')
    display(session_status_counts)
    print('Subjects with all required sessions')
    display(session_combination_summary.loc[session_combination_summary['n_missing_sessions'].eq(0)])
    print('Subjects by exact missing-session combination (including zero counts)')
    display(session_combination_summary.loc[session_combination_summary['n_missing_sessions'].gt(0)])
    print('Individual subject session status')
    display(subject_session_status)


Subject counts by session status
Subjects with all required sessions
Subjects by exact missing-session combination (including zero counts)
Individual subject session status


status,All sessions,Missing sessions,Total subjects
group,,,
Patient,31,11,42
HC,30,13,43


,group,status,n_missing_sessions,missing_sessions,n_subjects,subjects
0,Patient,All sessions,0,None,31,"sub-102, sub-104, sub-105, sub-106, sub-108, sub-113, sub-117, sub-119, sub-121, sub-123, sub-124, sub-125, sub-127, sub-130, sub-132, sub-133, sub-134, sub-135, sub-136, sub-138, sub-139, sub-140, sub-141, sub-143, sub-144, sub-145, sub-146, sub-147, sub-148, sub-150, sub-153"
16,HC,All sessions,0,None,30,"sub-301, sub-302, sub-303, sub-304, sub-305, sub-307, sub-308, sub-309, sub-310, sub-311, sub-312, sub-313, sub-314, sub-315, sub-317, sub-319, sub-320, sub-323, sub-324, sub-325, sub-326, sub-327, sub-328, sub-329, sub-330, sub-331, sub-332, sub-335, sub-336, sub-337"


,group,status,n_missing_sessions,missing_sessions,n_subjects,subjects
1,Patient,Missing sessions,1,baseline,0,—
2,Patient,Missing sessions,1,repeatbaseline,0,—
3,Patient,Missing sessions,1,t6,1,sub-110
4,Patient,Missing sessions,1,t12,1,sub-158
5,Patient,Missing sessions,2,"baseline, repeatbaseline",0,—
6,Patient,Missing sessions,2,"baseline, t6",0,—
7,Patient,Missing sessions,2,"baseline, t12",0,—
8,Patient,Missing sessions,2,"repeatbaseline, t6",0,—
9,Patient,Missing sessions,2,"repeatbaseline, t12",0,—
10,Patient,Missing sessions,2,"t6, t12",7,"sub-111, sub-160, sub-161, sub-162, sub-163, sub-164, sub-165"


,group,subID,status,n_expected_sessions,n_present_expected_sessions,present_sessions,missing_sessions,unexpected_sessions
0,HC,sub-301,All sessions,3,3,"baseline, repeatbaseline, t12",,
1,HC,sub-302,All sessions,3,3,"baseline, repeatbaseline, t12",,
2,HC,sub-303,All sessions,3,3,"baseline, repeatbaseline, t12",,
3,HC,sub-304,All sessions,3,3,"baseline, repeatbaseline, t12",,
4,HC,sub-305,All sessions,3,3,"baseline, repeatbaseline, t12",,
5,HC,sub-307,All sessions,3,3,"baseline, repeatbaseline, t12",,
6,HC,sub-308,All sessions,3,3,"baseline, repeatbaseline, t12",,
7,HC,sub-309,All sessions,3,3,"baseline, repeatbaseline, t12",,
8,HC,sub-310,All sessions,3,3,"baseline, repeatbaseline, t12",,
9,HC,sub-311,All sessions,3,3,"baseline, repeatbaseline, t12",,
